# 简单的代码分级评估

在本节课中，我们首先来看一个非常简单的代码分级评估示例，然后在下一节课中介绍一个更现实的提示词。我们将遵循此图所示的流程：



大致步骤如下：
1. 首先定义我们的评估测试集
1. 编写我们的初始提示词
2. 通过评估流程运行并获得分数
3. 根据评估结果修改提示词
4. 通过评估流程运行修改后的提示词，希望获得更好的分数！

让我们尝试遵循这个流程！

---

## 我们的输入数据

我们将对一个评估进行分级，要求 Claude 成功识别动物有多少条腿。在未来的课程中，我们会看到更复杂和更现实的提示词及评估，但这里我们特意保持简单，专注于实际的评估过程。

第一步是编写包含输入和相应标准答案的评估数据集。我们使用这个简单的字典列表，每个字典都有一个 `animal_statement` 和 `golden_answer` 键：

In [1]:
eval_data = [
    {"animal_statement": "The animal is a human.", "golden_answer": "2"},
    {"animal_statement": "The animal is a snake.", "golden_answer": "0"},
    {"animal_statement": "The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.", "golden_answer": "5"},
    {"animal_statement": "The animal is a dog.", "golden_answer": "4"},
    {"animal_statement": "The animal is a cat with two extra legs.", "golden_answer": "6"},
    {"animal_statement": "The animal is an elephant.", "golden_answer": "4"},
    {"animal_statement": "The animal is a bird.", "golden_answer": "2"},
    {"animal_statement": "The animal is a fish.", "golden_answer": "0"},
    {"animal_statement": "The animal is a spider with two extra legs", "golden_answer": "10"},
    {"animal_statement": "The animal is an octopus.", "golden_answer": "8"},
    {"animal_statement": "The animal is an octopus that lost two legs and then regrew three legs.", "golden_answer": "9"},
    {"animal_statement": "The animal is a two-headed, eight-legged mythical creature.", "golden_answer": "8"},
]

请注意，有些评估问题有点棘手，比如这个：
> The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.

这在后面会很重要！

---

## 我们的初始提示词
接下来，我们定义初始提示词。下面的函数接收一个动物陈述，并返回一个格式正确的消息列表，包含我们的第一个提示词：

In [2]:
def build_input_prompt(animal_statement):
    user_content = f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.
    
    Here is the animal statement.
    <animal_statement>{animal_statement}</animal_statement>
    
    How many legs does the animal have? Please respond with a number"""

    messages = [{'role': 'user', 'content': user_content}]
    return messages

让我们用 `eval` 数据集中的第一个元素快速测试一下：

In [3]:
build_input_prompt(eval_data[0]['animal_statement'])

[{'role': 'user',
  'content': 'You will be provided a statement about an animal and your job is to determine how many legs that animal has.\n    \n    Here is the animal statement.\n    <animal_statement>The animal is a human.</animal_statement>\n    \n    How many legs does the animal have? Please respond with a number'}]

接下来，我们将编写一个简单的函数，接收消息列表并将其发送到 Anthropic API：

In [5]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()

MODEL_NAME = "claude-3-haiku-20240307"

def get_completion(messages):
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=200,
        messages=messages
    )
    return response.content[0].text

让我们用 `eval_data` 列表中的第一个条目测试一下，它包含以下动物陈述：
```
'The animal is a human.'
```

In [6]:
full_prompt = build_input_prompt(eval_data[0]['animal_statement'])
get_completion(full_prompt)

'2'

我们得到 `2` 作为回复，通过了肉眼看测试！人类通常有两条腿。下一步是构建并运行完整的评估，使用 `eval_data` 集中的全部 12 个条目。

---

## 编写评估逻辑

我们首先将 `eval_data` 列表中的每个输入与提示词模板组合，然后将生成的"完成"提示词传递给模型，并收集返回的所有输出：

In [93]:

outputs = [get_completion(build_input_prompt(question['animal_statement'])) for question in eval_data]


让我们快速查看返回的内容：

In [94]:
outputs

['2',
 '0',
 '5',
 '4',
 '6',
 '4',
 'Based on the provided animal statement, "The animal is a bird.", the animal has 2 legs.\n\nResponse: 2',
 '0',
 '8',
 'An octopus has 8 legs.',
 '5',
 '8']

我们马上就能看出提示词需要改进，因为有些回答不是纯数字！让我们仔细看看结果以及每个对应的标准答案：

In [95]:
for output, question in zip(outputs, eval_data):
    print(f"Animal Statement: {question['animal_statement']}\nGolden Answer: {question['golden_answer']}\nOutput: {output}\n")

Animal Statement: The animal is a human.
Golden Answer: 2
Output: 2

Animal Statement: The animal is a snake.
Golden Answer: 0
Output: 0

Animal Statement: The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.
Golden Answer: 5
Output: 5

Animal Statement: The animal is a dog.
Golden Answer: 4
Output: 4

Animal Statement: The animal is a cat with two extra legs.
Golden Answer: 6
Output: 6

Animal Statement: The animal is an elephant.
Golden Answer: 4
Output: 4

Animal Statement: The animal is a bird.
Golden Answer: 2
Output: Based on the provided animal statement, "The animal is a bird.", the animal has 2 legs.

Response: 2

Animal Statement: The animal is a fish.
Golden Answer: 0
Output: 0

Animal Statement: The animal is a spider with two extra legs
Golden Answer: 10
Output: 8

Animal Statement: The animal is an octopus.
Golden Answer: 8
Output: An octopus has 8 legs.

Animal Statement: The animal is an octopus that lost two legs a

这个数据集很小，我们可以轻松扫描结果找到有问题的回复，但让我们系统地对结果进行评分：

In [97]:
def grade_completion(output, golden_answer):
    return output == golden_answer

grades = [grade_completion(output, question['golden_answer']) for output, question in zip(outputs, eval_data)]
print(f"Score: {sum(grades)/len(grades)*100}%")

Score: 66.66666666666666%


我们现在有了基线分数！在这种情况下，我们的初始提示词准确率为 66.6%。扫描以上结果后，我们当前输出似乎有两个明显的问题：

### 问题 1：输出格式问题
我们的目标是编写一个能产生数字输出的提示词。但有些输出不是数字：

```
Animal Statement: The animal is a bird.
Golden Answer: 2
Output: Based on the provided animal statement, "The animal is a bird.", the animal has 2 legs.
```
我们可以通过一些提示词技巧来修复这个问题！

### 问题 2：答案错误

此外，有些答案完全错误：

```
Animal Statement: The animal is an octopus that lost two legs and then regrew three legs.
Golden Answer: 9
Output: 5
```

以及

```
Animal Statement: The animal is a spider with two extra legs
Golden Answer: 10
Output: 8
```
这些输入有点"棘手"，似乎给模型带来了一些问题。我们也将尝试通过提示词来解决这个问题！

---

## 我们的第二次尝试

现在我们用初始提示词获得了一些基线性能，让我们尝试改进提示词，看看评估分数是否提高。我们首先解决模型有时输出额外文本而不仅仅是数字的问题。下面是第二个提示词生成函数：

In [98]:
def build_input_prompt2(animal_statement):
    user_content = f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.
    
    Here is the animal statement.
    <animal_statement>{animal_statement}</animal_statement>
    
    How many legs does the animal have? Respond only with a numeric digit, like 2 or 6, and nothing else."""

    messages = [{'role': 'user', 'content': user_content}]
    return messages

提示词的关键添加是这行：

> Respond only with a numeric digit, like 2 or 6, and nothing else.

让我们用这个新提示词测试每个输入：

In [99]:
outputs2 = [get_completion(build_input_prompt2(question['animal_statement'])) for question in eval_data]

我们快速查看输出：

In [101]:
outputs2

['2', '0', '6', '4', '6', '4', '2', '0', '8', '8', '5', '8']

我们现在得到纯数字输出了！让我们仔细看看结果：

In [102]:
for output, question in zip(outputs2, eval_data):
    print(f"Animal Statement: {question['animal_statement']}\nGolden Answer: {question['golden_answer']}\nOutput: {output}\n")

Animal Statement: The animal is a human.
Golden Answer: 2
Output: 2

Animal Statement: The animal is a snake.
Golden Answer: 0
Output: 0

Animal Statement: The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.
Golden Answer: 5
Output: 6

Animal Statement: The animal is a dog.
Golden Answer: 4
Output: 4

Animal Statement: The animal is a cat with two extra legs.
Golden Answer: 6
Output: 6

Animal Statement: The animal is an elephant.
Golden Answer: 4
Output: 4

Animal Statement: The animal is a bird.
Golden Answer: 2
Output: 2

Animal Statement: The animal is a fish.
Golden Answer: 0
Output: 0

Animal Statement: The animal is a spider with two extra legs
Golden Answer: 10
Output: 8

Animal Statement: The animal is an octopus.
Golden Answer: 8
Output: 8

Animal Statement: The animal is an octopus that lost two legs and then regrew three legs.
Golden Answer: 9
Output: 5

Animal Statement: The animal is a two-headed, eight-legged mythic

实际的数字答案仍然存在明显问题，比如这个：

```
Animal Statement: The animal is a spider with two extra legs
Golden Answer: 10
Output: 8
```

在处理那个问题之前，让我们获得一个官方分数，看看我们的性能（希望）是否提高了：

In [103]:
grades = [grade_completion(output, question['golden_answer']) for output, question in zip(outputs2, eval_data)]
print(f"Score: {sum(grades)/len(grades)*100}%")

Score: 75.0%


我们的分数提高了一点！**注意：这个数据集很小，所以请对这些结果持保留态度**

---

## 我们的第三次尝试

接下来，让我们解决我们在错误输出中看到的逻辑问题，例如：

```
Animal Statement: The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.
Golden Answer: 5
Output: 6
```
我们可以采用的一种技术是思维链提示词，我们给 Claude 具体的指令，在最终生成答案之前先推理出答案。现在我们有了评估框架，我们可以测试看看思维链提示词是否真的有区别！

让我们编写一个新提示词，要求模型在 `<thinking>` 标签内"说出来"。这会让我们的逻辑稍微复杂一些，因为我们需要一种方便的方法来提取模型的最终答案。我们将指示模型也将最终答案包含在 `<answer>` 标签中，这样我们就可以轻松提取"最终"的数字答案：

In [105]:
def build_input_prompt3(animal_statement):
    user_content = f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.
    
    Here is the animal statement.
    <animal_statement>{animal_statement}</animal_statement>
    
    How many legs does the animal have? 
    Start by reasoning about the numbers of legs the animal has, thinking step by step inside of <thinking> tags.  
    Then, output your final answer inside of <answer> tags. 
    Inside the <answer> tags return just the number of legs as an integer and nothing else."""

    messages = [{'role': 'user', 'content': user_content}]
    return messages

让我们用这个新版本的提示词收集输出：

In [109]:
outputs3 = [get_completion(build_input_prompt3(question['animal_statement'])) for question in eval_data]

现在让我们看看一些输出：

In [110]:
for output, question in zip(outputs3, eval_data):
    print(f"Animal Statement: {question['animal_statement']}\nGolden Answer: {question['golden_answer']}\nOutput: {output}\n")

Animal Statement: The animal is a human.
Golden Answer: 2
Output: <thinking>
The animal is a human, and based on this information, we can reasonably conclude that a human has 2 legs. Humans are bipedal, meaning they have two legs that they use for locomotion and standing upright. This is a characteristic of the human species.
</thinking>

<answer>2</answer>

Animal Statement: The animal is a snake.
Golden Answer: 0
Output: <thinking>
The animal stated in the given statement is a snake. Snakes are known to be legless reptiles, as they do not have any legs. They move by slithering on the ground using their body and scales.
</thinking>

<answer>0</answer>

Animal Statement: The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.
Golden Answer: 5
Output: Here is my step-by-step reasoning:
<thinking>
1. The initial statement says the fox lost a leg.
2. But then the fox "magically grew back the leg he lost and a mysterious extra leg on top 

这是我们得到的响应示例：

```
Animal Statement: The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.
Golden Answer: 5
Output: Here is my step-by-step reasoning:
<thinking>
1. The initial statement says the fox lost a leg.
2. But then the fox "magically grew back the leg he lost and a mysterious extra leg on top of that."
3. This means the fox originally had 4 legs, lost 1 leg, and then grew back the lost leg plus an extra leg, for a total of 5 legs.
</thinking>
<answer>5</answer>
```

逻辑看起来有所改善，至少在这个特定例子中是这样。现在我们需要专注于使这个提示词可以"评分"。在评分过程之前，我们需要提取 `answer` 标签之间的数字。

下面是一个提取两个 `<answer>` 标签之间文本的函数：

In [111]:
import re
def extract_answer(text):
    pattern = r'<answer>(.*?)</answer>'
    match = re.search(pattern, text)
    if match:
        return match.group(1)
    else:
        return None

接下来，让我们从最新一批输出中提取答案：

In [112]:
extracted_outputs3 = [extract_answer(output) for output in outputs3]

In [113]:
extracted_outputs3

['2', '0', '5', '4', '6', '4', '2', '0', '10', '8', '9', '8']

接下来，让我们获取分数，看看在提示词中添加思维链是否有区别！

In [114]:
grades3 = [grade_completion(output, question['golden_answer']) for output, question in zip(extracted_outputs3, eval_data)]
print(f"Score: {sum(grades3)/len(grades3)*100}%")

Score: 100.0%


我们将分数提高到了 100%！

我们的评估让我们有信心相信对提示词所做的更改实际上产生了更好的输出。这是一个使用精确匹配评分的简单示例，但在下一节课中，我们将看一个稍微复杂一些的例子。